# NKI Puzzles Release — NKI Beta 2 (For Students)

This release notebook is the student-facing classroom version for the NKI Beta 2 single-tile path.
It keeps the workflow simple: install from GitHub, run the demo, then complete the first three puzzles.

Included in this release:
- **Beta 2 Demo**: a tiny sanity check to confirm the environment works.
- **Puzzle 01**: constant add.
- **Puzzle 02**: outer vector add.
- **Puzzle 03**: single-tile matmul.

Notes:
- This notebook defaults to the GitHub install path for `triton-viz`.
- The visualizer LLM setup cell defaults to `max_tokens = 40960`.
- `pre_trace` is enabled by default.
- Later puzzles will be added in a future release.


In [ ]:
# Install deps from GitHub
GITHUB_TRITON_VIZ_REF = "jialiang/forclass"
EXTRA_INDEX_URL = "https://pip.repos.neuron.amazonaws.com"
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cpu"

import subprocess
import sys

install_target = (
    "triton-viz[nki] @ "
    f"git+https://github.com/Deep-Learning-Profiling-Tools/triton-viz.git@{GITHUB_TRITON_VIZ_REF}"
)
cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    install_target,
    "--extra-index-url",
    EXTRA_INDEX_URL,
]

print("Installing:", install_target)
subprocess.check_call(cmd)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "--index-url",
        TORCH_INDEX_URL,
    ]
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxtyping"])


Installing: triton-viz[nki] @ git+https://github.com/Deep-Learning-Profiling-Tools/triton-viz.git@jialiang/forclass


In [ ]:
from __future__ import annotations

import importlib.util
import inspect
import os
import tempfile
import textwrap
import types
from pathlib import Path
from typing import Any

import torch

os.environ["NKI_PUZZLES_USE_TRITON_VIZ_INTERPRETER"] = "1"

TRITON_VIZ_INTERPRETER_ENV = "NKI_PUZZLES_USE_TRITON_VIZ_INTERPRETER"
NKI_PUZZLES_PRE_TRACE_DEFAULT = "1"


def require_nki_beta2() -> tuple[Any, Any, Any]:
    import nki
    import nki.isa as nisa
    import nki.language as nl

    return nki, nisa, nl


def _torch_to_writable_numpy(t: torch.Tensor) -> Any:
    import numpy as np

    a = t.detach().cpu().contiguous().numpy()
    if not a.flags.writeable:
        a = a.copy()
    return a


def _serialize_kernel_constant(name: str, value: Any) -> str:
    try:
        import numpy as np
    except ModuleNotFoundError:
        np = None

    if np is not None and isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (int, float, bool, str, type(None), tuple, list, dict)):
        return f"{name} = {value!r}\n"
    raise TypeError(
        f"Unsupported notebook kernel closure value for pre_trace: {name}={type(value).__name__}"
    )


def _materialize_notebook_kernel(kernel: Any) -> tuple[Any, tempfile.TemporaryDirectory[str]]:
    source = textwrap.dedent(inspect.getsource(kernel))
    closure_vars = inspect.getclosurevars(kernel).nonlocals
    module_lines = [
        "import nki.isa as nisa\n",
        "import nki.language as nl\n",
        "\n",
    ]
    for name, value in closure_vars.items():
        if isinstance(value, types.ModuleType):
            continue
        module_lines.append(_serialize_kernel_constant(name, value))
    if closure_vars:
        module_lines.append("\n")
    module_lines.append(source)

    temp_dir = tempfile.TemporaryDirectory(prefix="nki_nb_kernel_")
    module_name = f"_nki_nb_kernel_{os.getpid()}_{abs(hash((kernel.__name__, id(kernel))))}"
    module_path = Path(temp_dir.name) / f"{module_name}.py"
    module_path.write_text("".join(module_lines))

    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        temp_dir.cleanup()
        raise RuntimeError(f"Failed to materialize notebook kernel: {kernel.__name__}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return getattr(module, kernel.__name__), temp_dir


def _run_triton_viz_beta2(kernel: Any, *args: Any, grid: tuple[int, ...] = (1,)) -> Any:
    import triton_viz

    require_nki_beta2()
    triton_viz.clear()
    pre_trace = os.environ.get("NKI_PUZZLES_PRE_TRACE", NKI_PUZZLES_PRE_TRACE_DEFAULT) == "1"
    orig_args = list(args)
    kernel_args = []
    torch_back = []
    for arg in args:
        if isinstance(arg, torch.Tensor):
            arr = _torch_to_writable_numpy(arg)
            torch_back.append((arg, arr))
            kernel_args.append(arr)
        else:
            kernel_args.append(arg)

    traced_kernel = kernel
    temp_dir = None
    if pre_trace and (
        kernel.__module__ == "__main__" or "<locals>" in getattr(kernel, "__qualname__", "")
    ):
        traced_kernel, temp_dir = _materialize_notebook_kernel(kernel)

    try:
        traced = triton_viz.trace("tracer", backend="nki_beta2")(traced_kernel)
        traced[grid](*kernel_args, pre_trace=pre_trace)
    finally:
        if temp_dir is not None:
            temp_dir.cleanup()

    for t_tensor, arr in torch_back:
        t_tensor.copy_(torch.from_numpy(arr))
    for a in reversed(orig_args):
        if isinstance(a, torch.Tensor):
            return a
    return None


def run_beta2_kernel(kernel: Any, *args: Any, grid: tuple[int, ...] = (1,)) -> Any:
    if os.environ.get(TRITON_VIZ_INTERPRETER_ENV) == "1":
        return _run_triton_viz_beta2(kernel, *args, grid=grid)

    nki, _, _ = require_nki_beta2()
    import torch_xla

    device = torch_xla.device()
    device_args = []
    for arg in args:
        if isinstance(arg, torch.Tensor):
            device_args.append(arg.to(device))
        else:
            device_args.append(arg)
    compiled_kernel = nki.jit(kernel, platform_target="trn1")
    out = compiled_kernel[grid](*device_args)
    torch_xla.sync()
    return out.cpu() if isinstance(out, torch.Tensor) else out


In [ ]:
def seed_all(seed: int = 0) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def allclose(got: torch.Tensor, ref: torch.Tensor, atol=1e-5, rtol=1e-5):
    if got.shape != ref.shape:
        return False, float("inf"), float("inf")
    ok = torch.allclose(got, ref, atol=atol, rtol=rtol)
    diff = (got - ref).abs()
    max_abs = float(diff.max().item()) if diff.numel() else 0.0
    denom = ref.abs().clamp_min(1e-12)
    max_rel = float((diff / denom).max().item()) if diff.numel() else 0.0
    return ok, max_abs, max_rel


def viz_launch(port: int = 8765, share: bool = True):
    import time
    import triton_viz
    from triton_viz.visualizer.interface import stop_server

    stop_server(port)
    time.sleep(1.0)

    triton_viz.launch(share=share, port=port)
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        pass


print("Prelude OK (nki_beta2 CPU interpreter enabled).")


In [ ]:
# Optional: visualizer LLM setup (edit and run before viz_launch)
import triton_viz

_LL_CONFIG_PATH = ""
_LL_API_KEY = ""
_LL_MODEL = ""
_LL_BASE_URL = ""
_LL_MAX_TOKENS = 40960

_applied = []
if _LL_CONFIG_PATH:
    triton_viz.setup_llm(config_path=_LL_CONFIG_PATH)
    _applied.append("config_path")
if _LL_API_KEY:
    triton_viz.setup_llm(api_key=_LL_API_KEY)
    _applied.append("api_key")
if _LL_MODEL:
    triton_viz.setup_llm(model=_LL_MODEL)
    _applied.append("model")
if _LL_BASE_URL:
    triton_viz.setup_llm(base_url=_LL_BASE_URL)
    _applied.append("base_url")
if _LL_MAX_TOKENS is not None:
    triton_viz.setup_llm(max_tokens=_LL_MAX_TOKENS)
    _applied.append("max_tokens")

print("setup_llm:", ", ".join(_applied) or "(nothing set)")


## Beta 2 Demo

A very small sanity check to confirm that the NKI Beta 2 CPU interpreter is working.

This is not one of the main puzzles.


In [ ]:
def demo_ref(x: torch.Tensor, c: float) -> torch.Tensor:
    return x + c


def demo_nki(x: torch.Tensor, c: float) -> torch.Tensor:
    _, nisa, nl = require_nki_beta2()
    x2 = x.reshape(-1, 1).contiguous()
    y2 = torch.empty_like(x2)
    c_val = float(c)

    def kernel(inp, out, c_val):
        rows, one = inp.shape
        assert one == 1
        tile = nl.ndarray((rows, 1), dtype=inp.dtype, buffer=nl.sbuf)
        out_tile = nl.ndarray((rows, 1), dtype=out.dtype, buffer=nl.sbuf)
        nisa.dma_copy(dst=tile, src=inp[:, :])
        nisa.tensor_scalar(dst=out_tile, data=tile, op0=nl.add, operand0=c_val)
        nisa.dma_copy(dst=out[:, :], src=out_tile)
        return out

    y2 = run_beta2_kernel(kernel, x2, y2, c_val)
    return y2.reshape_as(x)


seed_all(0)
x = torch.arange(8, dtype=torch.float32)
c = 2.0
y_ref = demo_ref(x, c)
y = demo_nki(x, c)
ok, max_abs, max_rel = allclose(y, y_ref)
print("demo:", ok, "max_abs:", max_abs, "max_rel:", max_rel)
# viz_launch()


## Main Path

The main path intentionally avoids blocked / tiled versions at first. The goal is to teach the core NKI Beta 2 programming model needed for the final project before introducing optimization details.


## Puzzle 01: Constant Add

Given a 1D tensor `x[N0]` and a scalar constant `c`, compute

- `y[i] = x[i] + c`

Keep this first puzzle as simple as possible: a **single-tile** elementwise add with no explicit blocking loop.


In [ ]:
def ref_constant_add(x: torch.Tensor, c: float) -> torch.Tensor:
    return x + c


def nki_constant_add(x: torch.Tensor, c: float) -> torch.Tensor:
    _, nisa, nl = require_nki_beta2()
    x2 = x.reshape(-1, 1).contiguous()
    y2 = torch.empty_like(x2)
    c_val = float(c)

    def kernel(inp, out, c_val):
        n, one = inp.shape
        assert one == 1
        assert n <= nl.tile_size.pmax
        input_tile = nl.ndarray((n,1), dtype=inp.dtype, buffer=nl.sbuf)
        output_tile = nl.ndarray((n,1), dtype=out.dtype, buffer=nl.sbuf)
        nisa.dma_copy(dst = input_tile, src=inp[:,:])
        nisa.tensor_scalar(dst= output_tile, data = input_tile, op0 = nl.add, operand0 = c_val)
        nisa.dma_copy(dst=out[:,:], src=output_tile)
        return out
        # TODO(student):
        # 1. Allocate one input tile and one output tile in sbuf.
        # 2. Copy inp into the input tile.
        # 3. Use tensor_scalar with nl.add and the passed-in c_val.
        # 4. Copy the result tile back to out.


    y2 = run_beta2_kernel(kernel, x2, y2, c_val)
    return y2.reshape_as(x)


seed_all(0)
x = torch.randn((128,), dtype=torch.float32)
c = 1.25
y_ref = ref_constant_add(x, c)
y = nki_constant_add(x, c)
ok, max_abs, max_rel = allclose(y, y_ref)
print("p01:", ok, "max_abs:", max_abs, "max_rel:", max_rel)
viz_launch()


## Puzzle 02: Outer Vector Add

Given two 1D tensors

- `x[N0]`
- `y[N1]`

compute the outer add result

- `z[j, i] = x[i] + y[j]`

with output shape `[N1, N0]`.

Keep this version simple as well: treat it as a **single-tile 2D output** with no explicit blocking loops.


In [ ]:
def ref_outer_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return x[None, :] + y[:, None]


def nki_outer_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    _, nisa, nl = require_nki_beta2()
    n0 = int(x.shape[0])
    n1 = int(y.shape[0])
    x_row = x.reshape(1, n0).contiguous()
    y_row = y.reshape(1, n1).contiguous()
    ones_x = torch.ones((1, n0), dtype=x.dtype)
    ones_y = torch.ones((1, n1), dtype=y.dtype)
    out = torch.empty((n1, n0), dtype=x.dtype)

    def kernel(x1, y1, ox, oy, z):
        _, cols = x1.shape
        _, rows = y1.shape
        tile_m = nl.tile_size.gemm_stationary_fmax
        tile_n = nl.tile_size.gemm_moving_fmax
        assert rows <= tile_m
        assert cols <= tile_n

        x = nl.ndarray((1,n0), dtype=x.dtype, buffer=nl.sbuf)
        y = nl.ndarray((1,n1), dtype=y.dtype, buffer=nl.sbuf)
        ones_x_tile=  nl.ndarray((1,n0), dtype=ox.dtype, buffer=nl.sbuf)
        ones_y_tile = nl.ndarray((1,n1), dtype=oy.dtype, buffer=nl.sbuf)

        tx = nl.ndarray((n1,n0), dtype=x.dtype, buffer=nl.sbuf)
        ty = nl.ndarray((n1,n0), dtype=y.dtype, buffer=nl.sbuf)

        nisa.dma_copy(dst=x, src=x1)
        nisa.dma_copy(dst=y, src=y1)
        nisa.dma_copy(dst=ones_x_tile, src=ox)
        nisa.dma_copy(dst=ones_y_tile, src=oy)

        nisa.nc_matmul(dst=tx, stationary=x, moving=ones_y_tile)
        nisa.nc_matmul(dst=ty, stationary=y, moving=ones_x_tile)

        results = nisa.tensor_tensor(tx,ty,op=nl.copy_op.add)
        nisa.dma_copy(dst=z, src=results)

        # TODO(student):
        # 1. Allocate tiles for x, y, ones_x, ones_y, and the output intermediates.
        # 2. DMA copy x1 / y1 / ox / oy into sbuf.
        # 3. Use nc_matmul twice to broadcast x and y across the 2D output.
        # 4. Add the two partial results with tensor_tensor.
        # 5. Copy the final tile back to z.


    return run_beta2_kernel(kernel, x_row, y_row, ones_x, ones_y, out)


seed_all(0)
n0, n1 = 32, 32
x = torch.randn((n0,), dtype=torch.float32)
y = torch.randn((n1,), dtype=torch.float32)
z_ref = ref_outer_add(x, y)
z = nki_outer_add(x, y)
ok, max_abs, max_rel = allclose(z, z_ref)
print("p02:", ok, "max_abs:", max_abs, "max_rel:", max_rel)
# viz_launch()


## Puzzle 03: MatMul (GEMM)

Given:
- `a`: `[M, K]`
- `b`: `[K, N]`

compute

- `c = a @ b`

with output shape `[M, N]`.

This single-tile version assumes the whole problem fits inside one NKI matmul tile.
That keeps the main path focused on data layout and `nc_matmul`, without Python tile loops.


In [ ]:
def ref_matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return a @ b


def nki_matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    _, nisa, nl = require_nki_beta2()
    m, k = a.shape
    k2, n = b.shape
    assert k == k2
    a_t = a.transpose(0, 1).contiguous()
    c = torch.empty((m, n), dtype=a.dtype)

    def kernel(lhs_t, rhs, out):
        kk, mm = lhs_t.shape
        kk2, nn = rhs.shape
        assert kk == kk2
        assert mm <= nl.tile_size.gemm_stationary_fmax
        assert kk <= nl.tile_size.pmax
        assert nn <= nl.tile_size.gemm_moving_fmax

        # TODO(student):
        # 1. Allocate one full lhs tile, one rhs tile, one psum tile, and one output sbuf tile.
        # 2. DMA copy the full inputs into sbuf.
        # 3. Run nc_matmul once.
        # 4. Copy the result out through sbuf back to HBM.
        raise NotImplementedError("TODO: implement Puzzle 03 kernel")
        return out

    return run_beta2_kernel(kernel, a_t, b.contiguous(), c)


seed_all(0)
m, n, k = 64, 64, 128
a = torch.randn((m, k), dtype=torch.float32)
b = torch.randn((k, n), dtype=torch.float32)
c_ref = ref_matmul(a, b)
c = nki_matmul(a, b)
ok, max_abs, max_rel = allclose(c, c_ref)
print("p03:", ok, "max_abs:", max_abs, "max_rel:", max_rel)
# viz_launch()
